# 01 — Smart meter producer


In [ ]:
import hashlib, json, math, os, random, uuid
from base64 import b64encode
from datetime import datetime, timedelta, timezone
import oci

missing = [key for key in ('OCI_STREAM_ENDPOINT', 'OCI_STREAM_OCID', 'OCI_CONFIG_FILE') if not os.getenv(key)]
if missing:
    raise ValueError('Missing required environment variables: ' + ', '.join(missing))

def event(meter_number, interval_start):
    if interval_start.tzinfo is None:
        raise ValueError('INTERVAL_START_UTC must include a timezone')
    start = interval_start.astimezone(timezone.utc).replace(second=0, microsecond=0) - timedelta(minutes=interval_start.minute % 15)
    meter_id = f'MTR-{meter_number:06d}'
    rng = random.Random(int(hashlib.sha256(f'42|{meter_id}|{start.isoformat()}'.encode()).hexdigest()[:16], 16))
    temperature = 28 + 7 * math.sin((start.hour - 13) * math.pi / 12) + rng.uniform(-1.5, 1.5)
    load = .20 + .55 * max(0, math.sin((start.hour - 10) * math.pi / 12))
    kwh = round(max(.03, (load + .018 * max(0, temperature - 30)) * (.83 if start.weekday() >= 5 else 1) + rng.gauss(0, .045)), 4)
    tariff = 'PEAK' if 17 <= start.hour < 22 else 'SHOULDER' if 7 <= start.hour < 17 else 'OFF_PEAK'
    return {'schema_version':'1.0','event_id':str(uuid.uuid5(uuid.UUID('34d45e32-9867-4fcf-b3eb-c5b7fe8b9c28'),f'{meter_id}|{start.isoformat()}')),'event_type':'INTERVAL_READING','event_ts_utc':(start+timedelta(minutes=15,seconds=rng.randint(1,120))).isoformat(),'meter_id':meter_id,'device_id':f'DEV-{meter_number:06d}','service_point_id':f'SP-{meter_number:06d}','service_point_type':'ELECTRIC','interval_start_utc':start.isoformat(),'interval_end_utc':(start+timedelta(minutes=15)).isoformat(),'interval_minutes':15,'consumption_kwh':kwh,'voltage_v':round(230+rng.gauss(0,3),2),'current_a':round(max(0,kwh*4000/230+rng.gauss(0,.2)),3),'power_factor':round(min(1,max(.75,.96+rng.gauss(0,.015))),3),'temperature_c':round(temperature,2),'quality_code':'ACTUAL','tariff_code':tariff,'measurement_events':[],'device_events':['TAMPER_ALERT'] if rng.random()<.0005 else []}

meter_count = int(os.getenv('METER_COUNT', '1000'))
value = os.getenv('INTERVAL_START_UTC')
interval_start = datetime.fromisoformat(value) if value else datetime.now(timezone.utc)
events = [event(number, interval_start) for number in range(1, meter_count + 1)]
client = oci.streaming.StreamClient(oci.config.from_file(os.environ['OCI_CONFIG_FILE'], os.getenv('OCI_PROFILE','DEFAULT')), service_endpoint=os.environ['OCI_STREAM_ENDPOINT'])
entries = [oci.streaming.models.PutMessagesDetailsEntry(key=b64encode(row['meter_id'].encode()).decode(), value=b64encode(json.dumps(row,separators=(',',':')).encode()).decode()) for row in events]
response = client.put_messages(os.environ['OCI_STREAM_OCID'], oci.streaming.models.PutMessagesDetails(messages=entries))
failures = [entry.error_message for entry in response.data.entries if entry.error]
if failures:
    raise RuntimeError(f'OCI rejected {len(failures)} events: {failures[:3]}')
print(f'Published {len(events)} readings for {interval_start.isoformat()}')